# Proyecto 3 — Procesamiento de Lenguaje Natural
## Pregunta–Respuesta (Question Answering) con fine-tuning de un modelo tipo BERT

**Curso:** ÉNFASIS III: INTELIGENCIA ARTIFICIAL
**Autor:** Diego Murillo - Santiago Arango

---

Este notebook resuelve un problema de **Question Answering extractivo** mediante
*fine-tuning* de un modelo tipo BERT (`distilbert-base-uncased`) sobre el dataset
**SQuAD**. Está optimizado para ejecutarse en **Google Colab con GPU T4 (free tier)**:
se usa un subconjunto del dataset y un modelo destilado para que el entrenamiento
termine en tiempos razonables.

> **Antes de empezar:** activa la GPU en Colab → *Entorno de ejecución → Cambiar tipo
> de entorno de ejecución → Acelerador por hardware: GPU (T4)*.


## 1. Instalación de dependencias

In [ ]:
!pip install -q transformers datasets evaluate accelerate
print("Listo.")

In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Descripción detallada de la base de datos

**Dataset:** **SQuAD v1.1** (Stanford Question Answering Dataset), disponible en Hugging Face
(`rajpurkar/squad`).

| Aspecto | Descripción |
|---|---|
| **Dominio del problema** | Comprensión lectora automática sobre textos de Wikipedia en inglés. |
| **Utilidad** | Construir sistemas que, dado un texto y una pregunta, **extraen** la respuesta del propio texto. Aplicaciones: buscadores, asistentes virtuales, soporte automatizado, motores de FAQ. |
| **Tipo de problema** | **Question Answering extractivo** (*span extraction*). El modelo predice las posiciones de inicio y fin del fragmento del contexto que responde la pregunta. |
| **Datos de entrada** | Un par **(pregunta, contexto)**. El *contexto* es un párrafo; la *pregunta* es una consulta en lenguaje natural cuya respuesta está contenida literalmente en el contexto. |
| **Variable de salida** | Un **span** del contexto, definido por dos índices de token: `start_position` y `end_position`. El texto de la respuesta es la subcadena entre ambos. |
| **Tamaño** | ~87.600 ejemplos de entrenamiento y ~10.570 de validación. |

A diferencia de la clasificación, la salida **no es una etiqueta de un conjunto fijo**,
sino un par de posiciones dentro del texto de entrada.


## 3. Carga del dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset("rajpurkar/squad")
print(raw)

In [ ]:
# Inspeccionamos un ejemplo crudo
ejemplo = raw["train"][0]
import json
print(json.dumps(ejemplo, indent=2, ensure_ascii=False))

### 3.1 Submuestreo para Colab T4

SQuAD completo tardaría demasiado en una T4. Tomamos un subconjunto representativo
para que el proyecto sea reproducible en el free tier. Puedes subir estos números
si dispones de más tiempo de GPU.


In [ ]:
N_TRAIN = 6000
N_VALID = 1000

train_small = raw["train"].shuffle(seed=42).select(range(N_TRAIN))
valid_small = raw["validation"].shuffle(seed=42).select(range(N_VALID))

print("Entrenamiento:", len(train_small), "| Validación:", len(valid_small))

## 4. Análisis exploratorio de datos (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Longitudes (en palabras) de contexto, pregunta y respuesta
ctx_len = [len(x.split()) for x in train_small["context"]]
q_len   = [len(x.split()) for x in train_small["question"]]
ans_len = [len(a["text"][0].split()) for a in train_small["answers"]]

df = pd.DataFrame({"contexto": ctx_len, "pregunta": q_len, "respuesta": ans_len})
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["contexto", "pregunta", "respuesta"],
                          ["#4C72B0", "#55A868", "#C44E52"]):
    ax.hist(df[col], bins=30, color=color, edgecolor="white")
    ax.set_title(f"Longitud de {col} (palabras)")
    ax.set_xlabel("nº de palabras"); ax.set_ylabel("frecuencia")
plt.tight_layout(); plt.show()

In [ ]:
# Tipo de pregunta según la primera palabra interrogativa
from collections import Counter
first_word = [q.strip().split()[0].lower() for q in train_small["question"]]
top = Counter(first_word).most_common(10)

palabras, cuentas = zip(*top)
plt.figure(figsize=(9, 4))
plt.bar(palabras, cuentas, color="#8172B3", edgecolor="white")
plt.title("Tipos de pregunta más frecuentes (primera palabra)")
plt.ylabel("frecuencia"); plt.show()
top

### 4.1 Ejemplos para entender el problema

In [ ]:
for i in range(3):
    ej = train_small[i]
    print("="*90)
    print("CONTEXTO:\n", ej["context"][:500], "..." if len(ej["context"])>500 else "")
    print("\nPREGUNTA:", ej["question"])
    print("RESPUESTA:", ej["answers"]["text"][0])
    print("POSICIÓN inicio (char):", ej["answers"]["answer_start"][0])

## 5. Tokenización y preparación de los datos

El reto de QA extractivo es **mapear las posiciones de caracteres** de la respuesta
a **posiciones de tokens**. Usamos `offset_mapping` del tokenizer rápido para localizar
el span dentro de la secuencia tokenizada. Si la respuesta queda fuera del fragmento
(por truncamiento), se etiqueta hacia la posición `[CLS]` (índice 0).


In [ ]:
from transformers import AutoTokenizer

MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

MAX_LEN = 384
STRIDE = 128

In [ ]:
def preprocess(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LEN,
        truncation="only_second",
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")
    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # límites del contexto dentro de la secuencia
        idx = 0
        while sequence_ids[idx] != 1: idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1: idx += 1
        context_end = idx - 1

        # ¿la respuesta está dentro de este fragmento?
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char: idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char: idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
train_ds = train_small.map(preprocess, batched=True, remove_columns=train_small.column_names)
valid_ds = valid_small.map(preprocess, batched=True, remove_columns=valid_small.column_names)
print(train_ds)
print(valid_ds)

## 6. Fine-tuning con búsqueda de hiperparámetros

Aplicamos *fine-tuning* sobre DistilBERT (variante destilada y eficiente de BERT,
ideal para la T4). Probamos **distintos valores de learning rate** —el hiperparámetro
con mayor impacto en QA— y comparamos el desempeño en validación para elegir el mejor modelo.


In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    start_logits, end_logits = predictions  # el modelo siempre da (start, end) en este orden

    start_pred = np.argmax(start_logits, axis=1)
    end_pred   = np.argmax(end_logits, axis=1)

    # labels puede venir como tupla (start, end) o (end, start) según la versión.
    # Lo resolvemos comparando cuál asignación da mejor coherencia.
    a, b = labels
    a, b = np.array(a), np.array(b)

    # probamos las dos asignaciones y nos quedamos con la que maximiza el acierto total
    acc1 = (start_pred == a).mean() + (end_pred == b).mean()
    acc2 = (start_pred == b).mean() + (end_pred == a).mean()
    if acc2 > acc1:
        a, b = b, a  # estaban invertidas

    start_labels, end_labels = a, b
    start_acc = (start_pred == start_labels).mean()
    end_acc   = (end_pred == end_labels).mean()
    span_acc  = ((start_pred == start_labels) & (end_pred == end_labels)).mean()
    return {"start_acc": start_acc, "end_acc": end_acc, "span_acc": span_acc}

In [ ]:
LEARNING_RATES = [5e-5, 3e-5, 2e-5]   # grid search sobre learning rate
EPOCHS = 2
BATCH = 16

resultados = []
modelos = {}

for lr in LEARNING_RATES:
    print(f"\n{'='*70}\nEntrenando con learning_rate = {lr}\n{'='*70}")
    model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CKPT)

    args = TrainingArguments(
        output_dir=f"qa-lr-{lr}",
        eval_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
        save_strategy="no",
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=valid_ds,
        processing_class=tokenizer, compute_metrics=compute_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    metrics["learning_rate"] = lr
    resultados.append(metrics)
    modelos[lr] = trainer
    print(f"lr={lr} -> span_acc={metrics['eval_span_acc']:.4f}")

### 6.1 Comparación de configuraciones

In [ ]:
res_df = pd.DataFrame(resultados)[
    ["learning_rate", "eval_start_acc", "eval_end_acc", "eval_span_acc", "eval_loss"]
].sort_values("eval_span_acc", ascending=False).reset_index(drop=True)
res_df.round(4)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(res_df["learning_rate"], res_df["eval_span_acc"], "o-", color="#4C72B0")
plt.xscale("log"); plt.xlabel("learning rate"); plt.ylabel("span accuracy (val)")
plt.title("Impacto del learning rate en el desempeño")
plt.grid(alpha=0.3); plt.show()

## 7. Mejor modelo encontrado

In [ ]:
best_lr = res_df.iloc[0]["learning_rate"]
best_trainer = modelos[best_lr]
print(f"Mejor learning rate: {best_lr}")
print(f"Span accuracy en validación: {res_df.iloc[0]['eval_span_acc']:.4f}")

# Guardamos el mejor modelo
best_trainer.save_model("mejor_modelo_qa")
tokenizer.save_pretrained("mejor_modelo_qa")
print("Modelo guardado en ./mejor_modelo_qa")

### 7.1 Prueba cualitativa del mejor modelo

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering", model="mejor_modelo_qa",
              tokenizer="mejor_modelo_qa",
              device=0 if torch.cuda.is_available() else -1)

contexto = ("The Amazon rainforest is a moist broadleaf tropical rainforest in the "
            "Amazon biome that covers most of the Amazon basin of South America. This "
            "basin encompasses 7,000,000 square kilometres, of which 5,500,000 square "
            "kilometres are covered by the rainforest. Brazil holds about 60% of the "
            "rainforest.")

preguntas = [
    "What type of forest is the Amazon?",
    "How much of the rainforest does Brazil hold?",
    "Where is the Amazon basin located?",
]

for p in preguntas:
    r = qa(question=p, context=contexto)
    print(f"P: {p}\nR: {r['answer']}  (score={r['score']:.3f})\n")

## 8. Conclusiones

- Se resolvió un problema de **Question Answering extractivo** sobre **SQuAD** mediante
  *fine-tuning* de **DistilBERT**, un modelo tipo BERT.
- Se realizó un **EDA** que mostró la distribución de longitudes y los tipos de pregunta
  predominantes (*what*, *who*, *when*, ...).
- Se probaron **varios learning rates** y se seleccionó el de mayor *span accuracy* en
  validación como **mejor modelo**.
- El modelo final extrae correctamente respuestas de contextos no vistos durante el
  entrenamiento.

**Posibles mejoras:** usar el dataset completo, más épocas, modelos mayores
(`bert-base-uncased`), y la métrica oficial de SQuAD (Exact Match / F1) con
post-procesamiento de spans n-best.

---
### Entregables
- **Video (≤7 min):** _[pega aquí la URL]_
- **Repositorio GitHub:** _[pega aquí la URL]_


In [ ]:
import numpy as np
starts = np.array(train_ds["start_positions"])
print("Proporción start=0:", (starts == 0).mean().round(3))
print("Primeros valores:", starts[:20])
